# WRO pillar + parking-lot YOLO — one-click retrain (Team Blueprint)

**Before the first run (once):** `Runtime → Change runtime type → T4 GPU`, and add your Roboflow key as a Colab secret:
key icon on the left → **Add new secret** → name `ROBOFLOW_API_KEY`, value = your private API key → switch **Notebook access** on.

**Every retrain:** fill in cell 1 → `Runtime → Run all` → wait ~10 min → the model zip downloads by itself →
on the PC: `python tools/yolo/yolo.py deploy`.

Full protocol: `tools/yolo/PROTOCOL.md` in BlueprintPi.

In [ ]:
#@title 1. Settings { display-mode: "form" }
WORKSPACE = ""  #@param {type:"string"}
PROJECT = ""  #@param {type:"string"}
VERSION = 0  #@param {type:"integer"}
#@markdown `VERSION` 0 = the newest version. Generate the version in Roboflow first (no augmentations needed - YOLO augments).
BASE = "pillars26 (from GitHub)"  #@param ["pillars26 (from GitHub)", "last model saved to Drive", "yolo26n.pt (from scratch)"]
EPOCHS = 60  #@param {type:"slider", min:10, max:150, step:5}
#@markdown 60 epochs with early stop (patience 15) is ~6-10 min on a T4 for 500-1500 images. Starting from our own model converges fast.
IMGSZ = 416  #@param [320, 416, 480, 640] {type:"raw"}
#@markdown Keep 416: that is what the Pi runs at ~28 ms/frame (ncnn).
SAVE_TO_DRIVE = True  #@param {type:"boolean"}
#@markdown Drive keeps `wro_yolo/latest.pt` + every zip, so the next retrain can start from today's model.

import os, time
ROOT = os.environ.get("WRO_ROOT", "/content")
REPO_RAW = "https://github.com/josef-ami/BlueprintPi/raw/main"
MODEL_NAME = time.strftime("pillars_%Y%m%d_%H%M")
print("model name:", MODEL_NAME)

In [ ]:
#@title 2. Install (about 1 min)
import subprocess, sys
subprocess.run("nvidia-smi -L", shell=True)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "ultralytics==8.4.160", "roboflow",
                "onnx", "onnxslim", "onnxruntime"], check=True)
import torch
DEVICE = 0 if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    print("WARNING: no GPU - Runtime > Change runtime type > T4 GPU (CPU training takes ~10x longer)")

In [ ]:
#@title 3. Dataset from Roboflow (or an uploaded export zip)
import glob, os, shutil, zipfile, yaml
DS = os.path.join(ROOT, "ds")
key = None
try:
    from google.colab import userdata
    key = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    pass
if key and WORKSPACE and PROJECT:
    from roboflow import Roboflow
    proj = Roboflow(api_key=key).workspace(WORKSPACE).project(PROJECT)
    ver = VERSION or max(int(str(v.version).split("/")[-1]) for v in proj.versions())
    print(f"downloading {WORKSPACE}/{PROJECT} version {ver}")
    DS = proj.version(ver).download("yolov8", location=DS, overwrite=True).location
    DATASET_TAG = f"{PROJECT} v{ver}"
else:
    zips = glob.glob(os.path.join(ROOT, "*.zip"))
    if not zips:
        print("No Roboflow key/project - upload the Roboflow export zip (format: YOLOv8):")
        from google.colab import files
        up = files.upload()
        zips = [os.path.join(ROOT, n) for n in up]
    shutil.rmtree(DS, ignore_errors=True)
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(DS)
    DATASET_TAG = os.path.basename(zips[0])

y = yaml.safe_load(open(os.path.join(DS, "data.yaml")))
for k, sub in (("train", "train"), ("val", "valid"), ("test", "test")):
    p = os.path.join(DS, sub, "images")
    if os.path.isdir(p) and os.listdir(p):
        y[k] = p
    else:
        y.pop(k, None)
if "val" not in y:
    y["val"] = y["train"]
    print("WARNING: no valid split - validating on train. In Roboflow use a 80/15/5 split.")
DATA_YAML = os.path.join(ROOT, "data.yaml")
yaml.safe_dump(y, open(DATA_YAML, "w"))
names = y["names"] if isinstance(y["names"], list) else [y["names"][i] for i in sorted(y["names"])]
for k in ("train", "val"):
    print(f"{k:5s}: {len(os.listdir(y[k]))} images")
print("classes:", names)
assert any("RED" in n.upper() for n in names) and any("GREEN" in n.upper() for n in names), \
    "the dataset has no RED / GREEN class - wrong project?"
if not any("PARK" in n.upper() for n in names):
    print("NOTE: no PARKING LOT class in this version - the model will only find pillars")

In [ ]:
#@title 4. Starting weights
import urllib.request
BASE_PT = os.path.join(ROOT, "base.pt")
DRIVE_DIR = "/content/drive/MyDrive/wro_yolo"
if SAVE_TO_DRIVE or BASE.startswith("last"):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)
    except Exception as e:
        print("Drive not mounted:", e); SAVE_TO_DRIVE = False
if BASE.startswith("pillars26"):
    try:
        urllib.request.urlretrieve(f"{REPO_RAW}/models/pillars26/best.pt", BASE_PT)
    except Exception as e:
        print("GitHub download failed:", e, "- upload models/pillars26/best.pt from the PC:")
        from google.colab import files
        up = files.upload()
        shutil.move(os.path.join(ROOT, next(iter(up))), BASE_PT)
elif BASE.startswith("last"):
    shutil.copy(os.path.join(DRIVE_DIR, "latest.pt"), BASE_PT)
else:
    BASE_PT = "yolo26n.pt"
print("starting from", BASE_PT)

In [ ]:
#@title 5. Train (the long one: ~6-10 min on a T4)
from ultralytics import YOLO
t0 = time.time()
model = YOLO(BASE_PT)
model.train(data=DATA_YAML, imgsz=IMGSZ, epochs=EPOCHS, patience=15, batch=32 if DEVICE != "cpu" else 8,
            device=DEVICE, cache="ram", workers=4, cos_lr=True, close_mosaic=10,
            hsv_h=0.01,            # colour IS the class: keep hue shifts tiny
            project=os.path.join(ROOT, "runs"), name=MODEL_NAME, exist_ok=True, plots=True, seed=0)
BEST = str(model.trainer.best)
print(f"trained in {(time.time() - t0) / 60:.1f} min -> {BEST}")

In [ ]:
#@title 6. How good is it? (per class, on the valid split)
m = YOLO(BEST)
r = m.val(data=DATA_YAML, imgsz=IMGSZ, device=DEVICE, plots=False, verbose=False)
print(f"{'class':14s} {'P':>6s} {'R':>6s} {'mAP50':>6s}")
for i, c in enumerate(r.box.ap_class_index):
    print(f"{m.names[int(c)]:14s} {r.box.p[i]:6.3f} {r.box.r[i]:6.3f} {r.box.ap50[i]:6.3f}")
print(f"{'all':14s} {r.box.mp:6.3f} {r.box.mr:6.3f} {r.box.map50:6.3f}")
METRICS = {"mAP50": float(r.box.map50), "mAP50_95": float(r.box.map),
           "per_class_mAP50": {m.names[int(c)]: float(r.box.ap50[i]) for i, c in enumerate(r.box.ap_class_index)}}
if r.box.map50 < 0.85:
    print("\nWARNING: mAP50 < 0.85 - check the labels in Roboflow before trusting this model on the car")
try:
    from IPython.display import Image, display
    display(Image(os.path.join(os.path.dirname(os.path.dirname(BEST)), "confusion_matrix_normalized.png"), width=520))
except Exception:
    pass

In [ ]:
#@title 7. Export for the Pi (ncnn + onnx), check, zip, download
import json, onnxruntime as ort
onnx_path = YOLO(BEST).export(format="onnx", imgsz=IMGSZ, end2end=False)
ncnn_path = YOLO(BEST).export(format="ncnn", imgsz=IMGSZ, end2end=False)
shape = ort.InferenceSession(onnx_path).get_outputs()[0].shape
assert shape[1] == 4 + len(m.names), f"unexpected head {shape}: the Pi decoder needs raw (4+classes) x anchors"

pkg = os.path.join(ROOT, "export", MODEL_NAME)
shutil.rmtree(os.path.join(ROOT, "export"), ignore_errors=True)
os.makedirs(pkg)
shutil.copy(BEST, os.path.join(pkg, "best.pt"))
shutil.copy(onnx_path, os.path.join(pkg, "best.onnx"))
shutil.copytree(ncnn_path, os.path.join(pkg, "best_ncnn_model"),     # the Pi needs .param/.bin/metadata only
                ignore=shutil.ignore_patterns("__pycache__", "*.py", "*pnnx*"))
info = {"name": MODEL_NAME, "classes": m.names, "imgsz": IMGSZ, "dataset": DATASET_TAG,
        "base": BASE, "epochs": EPOCHS, "trained": time.strftime("%Y-%m-%d %H:%M"), **METRICS}
json.dump(info, open(os.path.join(pkg, "info.json"), "w"), indent=2)
ZIP = shutil.make_archive(os.path.join(ROOT, MODEL_NAME), "zip", os.path.join(ROOT, "export"))
print("packed", ZIP, f"({os.path.getsize(ZIP) / 1e6:.1f} MB)")
if SAVE_TO_DRIVE:
    shutil.copy(ZIP, DRIVE_DIR)
    shutil.copy(BEST, os.path.join(DRIVE_DIR, "latest.pt"))
    print("saved to Drive:", DRIVE_DIR)
try:
    from google.colab import files
    files.download(ZIP)
    print("\nNext, on the PC:  python tools/yolo/yolo.py deploy")
except Exception:
    print("not in Colab - zip is at", ZIP)